In [ ]:
# Install required libraries (if not already installed)
!pip install scikit-learn pandas numpy matplotlib seaborn joblib

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

In [ ]:
import os
print("Current directory:", os.getcwd())
print("\nFiles in current directory:")
print(os.listdir())

In [ ]:
from google.colab import files
uploaded = files.upload()  # This will open a file upload dialog
# Then select your IRIS.csv file from your computer

In [ ]:
import pandas as pd
import os

print("Files after upload:")
print(os.listdir())

# Now try loading the file
df = pd.read_csv('IRIS.csv')
print("\nFile loaded successfully!")
print(f"Shape: {df.shape}")

In [ ]:
# Load dataset
df = pd.read_csv('IRIS.csv')

# Display basic info
print("=== DATASET SHAPE ===")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n=== FIRST 5 ROWS ===")
print(df.head())

print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

In [ ]:
# 1. Class Distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df['species'].value_counts().plot(kind='bar', color=['skyblue', 'lightgreen', 'salmon'])
plt.title('Species Distribution')
plt.xlabel('Species')
plt.ylabel('Count')
plt.xticks(rotation=45)

# 2. Basic Statistics
plt.subplot(1, 2, 2)
df.describe().loc[['mean', 'std', 'min', 'max']].plot(kind='bar', ax=plt.gca())
plt.title('Feature Statistics')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 3. Scatter Plot (Sepal)
plt.figure(figsize=(8, 6))
for species in df['species'].unique():
    subset = df[df['species'] == species]
    plt.scatter(subset['sepal_length'], subset['sepal_width'],
                label=species, alpha=0.7, s=50)
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Sepal Width (cm)')
plt.title('Sepal Length vs Width')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 4. Box Plot
plt.figure(figsize=(10, 6))
df.boxplot(by='species', column=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])
plt.suptitle('')
plt.title('Feature Distributions by Species')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Encode species labels (text to numbers)
label_encoder = LabelEncoder()
df['species_encoded'] = label_encoder.fit_transform(df['species'])

# 2. Separate features (X) and target (y)
X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y = df['species_encoded']

# 3. Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=== DATA SPLIT ===")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

# 4. Standardize features (important for some algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("\nFeatures standardized (mean=0, std=1)")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Initialize models
knn_model = KNeighborsClassifier(n_neighbors=3)
logistic_model = LogisticRegression(random_state=42, max_iter=200)
tree_model = DecisionTreeClassifier(random_state=42, max_depth=3)

# Train models
print("Training K-Nearest Neighbors...")
knn_model.fit(X_train_scaled, y_train)

print("Training Logistic Regression...")
logistic_model.fit(X_train_scaled, y_train)

print("Training Decision Tree...")
tree_model.fit(X_train, y_train)  # Trees don't need scaling

print("\n=== ALL MODELS TRAINED ===")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score
import seaborn as sns

# Function to evaluate model
def evaluate_model(model, X_test_data, y_true, model_name, needs_scaling=True):
    # Make predictions
    if needs_scaling:
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')

    print(f"\n{'='*50}")
    print(f"{model_name.upper()} RESULTS")
    print(f"{'='*50}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    return accuracy

# Evaluate all models
accuracies = {}
accuracies['KNN'] = evaluate_model(knn_model, X_test_scaled, y_test, "K-Nearest Neighbors", needs_scaling=True)
accuracies['Logistic Regression'] = evaluate_model(logistic_model, X_test_scaled, y_test, "Logistic Regression", needs_scaling=True)
accuracies['Decision Tree'] = evaluate_model(tree_model, X_test, y_test, "Decision Tree", needs_scaling=False)

# Compare models
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
for model_name, acc in accuracies.items():
    print(f"{model_name:20} Accuracy: {acc:.4f}")

# Find best model
best_model_name = max(accuracies, key=accuracies.get)
print(f"\n🎯 BEST MODEL: {best_model_name} (Accuracy: {accuracies[best_model_name]:.4f})")

In [ ]:
import joblib

# Identify best model
if best_model_name == 'KNN':
    best_model = knn_model
    needs_scaling_for_inference = True
elif best_model_name == 'Logistic Regression':
    best_model = logistic_model
    needs_scaling_for_inference = True
else:
    best_model = tree_model
    needs_scaling_for_inference = False

# Save all necessary components
model_data = {
    'model': best_model,
    'scaler': scaler if needs_scaling_for_inference else None,
    'label_encoder': label_encoder,
    'feature_names': list(X.columns),
    'model_name': best_model_name,
    'needs_scaling': needs_scaling_for_inference
}

# Save to file
joblib.dump(model_data, 'best_iris_model.joblib')
print("✅ Model saved as: 'best_iris_model.joblib'")

In [ ]:
# Load the saved model
loaded_data = joblib.load('best_iris_model.joblib')
loaded_model = loaded_data['model']
loaded_scaler = loaded_data['scaler']
loaded_encoder = loaded_data['label_encoder']

# Function to make predictions
def predict_iris(sepal_length, sepal_width, petal_length, petal_width):
    """
    Predict iris species from measurements.

    Args:
        sepal_length, sepal_width, petal_length, petal_width (float): Measurements in cm

    Returns:
        str: Predicted species name
    """
    # Create array of features
    features = np.array([[sepal_length, sepal_width, petal_length, petal_width]])

    # Scale if needed
    if loaded_data['needs_scaling']:
        features = loaded_scaler.transform(features)

    # Make prediction
    prediction_num = loaded_model.predict(features)[0]
    species = loaded_encoder.inverse_transform([prediction_num])[0]

    return species

# Test with real examples
print("🧪 TESTING INFERENCE FUNCTION")
print("-" * 40)

# Example 1: Iris-setosa
result1 = predict_iris(5.1, 3.5, 1.4, 0.2)
print(f"Input: [5.1, 3.5, 1.4, 0.2] → Prediction: {result1}")

# Example 2: Iris-versicolor
result2 = predict_iris(6.0, 2.7, 4.5, 1.5)
print(f"Input: [6.0, 2.7, 4.5, 1.5] → Prediction: {result2}")

# Example 3: Iris-virginica
result3 = predict_iris(7.0, 3.2, 6.0, 2.5)
print(f"Input: [7.0, 3.2, 6.0, 2.5] → Prediction: {result3}")

print("\n✅ Inference working correctly!")

In [ ]:
# Download the saved model file
from google.colab import files

# List files to confirm model exists
import os
print("Files in current directory:")
for file in os.listdir():
    if 'joblib' in file or 'pkl' in file:
        print(f"✓ {file}")

# Download the model file
try:
    files.download('best_iris_model.joblib')
    print("✅ Model file downloaded successfully!")
except:
    print("❌ Model file not found. Let's save it first:")

    # Save model if not already saved
    import joblib
    # (Re-run your model saving code if needed)
    print("Run your model saving code again:")
    print('joblib.dump(model_data, "best_iris_model.joblib")')